# Горизонт передбачуваності в моделі Лоренца

**Автори:** Артем, Еван

## Фабула

У 1961 році Едвард Лоренц запустив спрощену модель атмосферної конвекції на комп'ютері LGP-30 і виявив, що два запуски з майже однаковими початковими умовами (відмінність у п'ятому десятковому знаку) через короткий час дають **принципово різні** прогнози. Так народилася теорія детермінованого хаосу і фраза «ефект метелика».

Це відкриття має конкретний практичний наслідок: **існує фундаментальна межа передбачуваності погоди** — не через обмеження комп'ютерів, а через саму природу динамічних рівнянь. Сьогодні ця межа складає 10–14 днів, і жодні покращення точності вимірювань її радикально не відсунуть.

У цьому проєкті ми дослідимо механізм цього явища на класичній системі Лоренца, поєднуючи **символьний аналіз стійкості** (локальна картина — нерухомі точки та власні значення) з **чисельним експериментом** (глобальна картина — розходження траєкторій у часі).

## Математична модель

Система Лоренца:
$$
\begin{cases}
\dot x = \sigma(y - x) \\
\dot y = x(\rho - z) - y \\
\dot z = xy - \beta z
\end{cases}
$$

Класичні параметри, для яких спостерігається хаос:
$$\sigma = 10, \quad \rho = 28, \quad \beta = 8/3.$$

Фізичний зміст змінних: $x$ — інтенсивність конвекції, $y$ — різниця температур між висхідним і низхідним потоками, $z$ — відхилення вертикального температурного профілю від лінійного.

## План дослідження

1. **Символьно** знайти всі нерухомі точки системи та визначити їхній тип через власні значення лінеаризації.
2. **Чисельно** проінтегрувати дві траєкторії з близькими початковими умовами та виміряти, як росте відстань між ними.
3. **Оцінити показник Ляпунова** $\lambda$ через лінійну регресію $\ln d(t)$ і порівняти з теоретичним значенням.
4. **Візуалізувати** результати інтерактивно (Plotly): атрактор у 3D з нанесеними нерухомими точками + семилогарифмічний графік розходження.

In [1]:
import numpy as np
from scipy.integrate import solve_ivp
import json

import plotly.graph_objects as go



Частина 2. Чисельна симуляція чутливості до початкових умов
Тепер запустимо дві траєкторії з мікроскопічно різними початковими умовами (різниця $10^{-6}$ у координаті $x$, тобто приблизно шум при 6-значній точності вимірювання). Для високоточного інтегрування використаємо метод Дорманда–Принса 8-го порядку (DOP853) з відносною точністю $10^{-10}$ — це набагато менше, ніж наше збурення, тож спостережуване розходження буде справжнім, а не числовим артефактом.

In [2]:
# Параметри як звичайні float для numpy/scipy
SIGMA, RHO, BETA = 10.0, 28.0, 8.0 / 3.0

def lorenz(t, state):
    xv, yv, zv = state
    return [
        SIGMA * (yv - xv),
        xv * (RHO - zv) - yv,
        xv * yv - BETA * zv,
    ]

# Прогрів: інтегруємо від [1, 1, 1] протягом 10 одиниць часу,
# щоб вийти на сам атрактор і позбутися перехідного процесу
burn = solve_ivp(lorenz, (0.0, 10.0), [1.0, 1.0, 1.0],
                 method="DOP853", rtol=1e-10, atol=1e-12)
p0 = burn.y[:, -1]
print(f"Точка на атракторі: ({p0[0]:.3f}, {p0[1]:.3f}, {p0[2]:.3f})")

# Дві близькі початкові умови
ic_A = p0.copy()
ic_B = p0 + np.array([1e-6, 0.0, 0.0])

# Основна симуляція на [0, 25]
T_MAX = 25.0
t_eval = np.linspace(0.0, T_MAX, 5001)
opts = dict(method="DOP853", rtol=1e-10, atol=1e-12, t_eval=t_eval)

sol_A = solve_ivp(lorenz, (0.0, T_MAX), ic_A, **opts)
sol_B = solve_ivp(lorenz, (0.0, T_MAX), ic_B, **opts)

print(f"Початкове збурення: d(0) = {np.linalg.norm(ic_A - ic_B):.1e}")

Точка на атракторі: (-4.903, -3.744, 24.691)
Початкове збурення: d(0) = 1.0e-06
